<a href="https://colab.research.google.com/github/nitin-rajesh/NLP-RL-Pipeline/blob/main/ProcessPromptNaturalInstructions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import json
import random
from ast import literal_eval

from datasets import Dataset



In [5]:
# !pip install bitsandbytes
# # !pip install transformers accelerate safetensors
# !pip install accelerate
# !pip install unsloth
# !pip install xformers


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
from huggingface_hub import login
from google.colab import userdata
# userdata.get('secretName')
# load from Colab Secrets
hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

# ensure all HF libs use this token
os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token
os.environ["HF_TOKEN"] = hf_token

print("HF auth complete.")


HF auth complete.


In [4]:
MODEL_NAME= ""
BATCH_SIZE = 4
INPUT_JSONL = "/content/drive/MyDrive/AdvNLP/natural_instructions_1M_promptgeneratedNew.jsonl"   # your existing jsonl
# OUTPUT_JSONL = "/content/drive/MyDrive/AdvNLP/natural_instructions_1M_promptgeneratedNew2.jsonl" # new file to store prompt-generation outputs
TARGET_COUNT = 1_000_000   # how many examples to process (set lower for a pilot)
SAVE_EVERY = 100
# Batching configuration
BATCH_SIZE = 4            # start with 8; increase to 16 if memory allows
MAX_NEW_TOKENS = 32000         # per-example generation length (adjust if needed)
TEMPERATURE = 0.3
TOP_P = 0.9

In [5]:
def extract_4_prompts_from_record(record):
    """
    Given one record (a dict loaded from your JSONL line), return a tuple:
      (orig_prompt, p1_prompt, p2_prompt, p3_prompt, sources)
    where `sources` is a list like ["orig","from_parsed","from_raw_regex","fallback"]
    The function does NOT read/write files — it's immediate and deterministic.
    """
    import json, re, time

    # ---- helpers (local to the cell) ----
    def _extract_json_from_text(text):
        if not text or not isinstance(text, str):
            return None
        text = text.strip()
        try:
            return json.loads(text)
        except Exception:
            pass
        # try to find a balanced { ... } block
        start = text.find("{")
        end = text.rfind("}")
        if start >= 0 and end > start:
            cand = text[start:end+1]
            try:
                return json.loads(cand)
            except Exception:
                # try balanced pairs
                brace = 0
                s = None
                for i,ch in enumerate(text):
                    if ch == "{":
                        brace += 1
                        if s is None:
                            s = i
                    elif ch == "}":
                        brace -= 1
                        if s is not None and brace == 0:
                            cand2 = text[s:i+1]
                            try:
                                return json.loads(cand2)
                            except Exception:
                                s = None
                return None
        return None

    _PROMPT_RE = re.compile(r'"prompt"\s*:\s*"(.*?)"', re.DOTALL)
    def _loose_extract_prompts(raw_text):
        if not raw_text or not isinstance(raw_text, str):
            return []
        # try JSON block first
        j = _extract_json_from_text(raw_text)
        if j and isinstance(j, dict):
            out = []
            p_list = j.get("prompts")
            if isinstance(p_list, list):
                for it in p_list:
                    if isinstance(it, dict) and "prompt" in it:
                        out.append(it["prompt"])
                    elif isinstance(it, str):
                        out.append(it)
            else:
                # look for dict entries that contain prompt keys
                for v in j.values():
                    if isinstance(v, dict) and "prompt" in v:
                        out.append(v["prompt"])
            if out:
                return out
        # regex fallback
        m = _PROMPT_RE.findall(raw_text)
        if m:
            # unescape common sequences
            return [bytes(s, "utf-8").decode("unicode_escape") for s in m]
        # looser pattern
        m2 = re.findall(r'prompt\s*[:=]\s*"(.*?)"', raw_text, re.DOTALL | re.IGNORECASE)
        if m2:
            return [bytes(s, "utf-8").decode("unicode_escape") for s in m2]
        return []

    def _make_fallbacks(definition, input_text):
        d = (definition or "").strip()
        i = (input_text or "").strip()
        p1 = f"{d} Input: {i} Provide a concise direct answer only, formatted as requested."
        p2 = f"Step-by-step: {d} Use numbered steps to extract facts from the Input then provide the final answer. Input: {i}"
        p3 = f"Context-aware: {d} Consider likely assumptions and world knowledge to disambiguate, then give the requested answer. Input: {i}"
        return [p1, p2, p3]

    # ---- main extraction logic ----
    definition = record.get("definition","") or ""
    input_text = record.get("input","") or record.get("inputs","") or record.get("text","") or ""
    orig_prompt = f"{definition.strip()}\n\nInput:\n{input_text.strip()}"

    # try parsed_json first
    gen = record.get("generation", {}) or {}
    parsed = gen.get("parsed_json")
    raw_text = gen.get("raw_text") or gen.get("raw_output") or None

    extracted = []
    sources = []

    if parsed and isinstance(parsed, dict) and not parsed.get("error_parse", False):
        p_list = parsed.get("prompts")
        if isinstance(p_list, list) and len(p_list) > 0:
            for item in p_list:
                if isinstance(item, dict) and "prompt" in item:
                    extracted.append(item["prompt"])
                    sources.append("from_parsed")
                elif isinstance(item, str):
                    extracted.append(item)
                    sources.append("from_parsed_str")
    # if we still need prompts, try raw_text JSON block extraction / regex
    if len(extracted) < 3 and raw_text:
        found = _loose_extract_prompts(raw_text)
        if found:
            for s in found:
                if len(extracted) >= 3: break
                extracted.append(s)
                sources.append("from_raw")
    # fill with fallbacks if necessary
    if len(extracted) < 3:
        fb = _make_fallbacks(definition, input_text)
        for s in fb:
            if len(extracted) >= 3: break
            extracted.append(s)
            sources.append("fallback")

    # ensure exactly 3 prompts
    while len(extracted) < 3:
        extracted.append("")
        sources.append("empty")

    # final outputs: orig + p1,p2,p3
    p1, p2, p3 = extracted[0], extracted[1], extracted[2]
    return orig_prompt, p1, p2, p3, ["orig"] + sources


In [22]:

import json, re, time
from tqdm.auto import tqdm

# INPUT_JSONL = "/content/drive/MyDrive/AdvNLP/natural_instructions_1M_promptgen.jsonl"

def extract_4_prompts_from_record(record):
    import json, re
    def _extract_json_from_text(text):
        if not text or not isinstance(text, str):
            return None
        text = text.strip()
        try:
            return json.loads(text)
        except Exception:
            pass
        start = text.find("{")
        end = text.rfind("}")
        if start >= 0 and end > start:
            cand = text[start:end+1]
            try:
                return json.loads(cand)
            except Exception:
                brace = 0
                s = None
                for i,ch in enumerate(text):
                    if ch == "{":
                        brace += 1
                        if s is None:
                            s = i
                    elif ch == "}":
                        brace -= 1
                        if s is not None and brace == 0:
                            cand2 = text[s:i+1]
                            try:
                                return json.loads(cand2)
                            except Exception:
                                s = None
                return None
        return None


    def _safe_unescape(s: str) -> str:
        # Try JSON-style unescape first (handles \n, \", \\ etc. safely)
        try:
            return json.loads(f'"{s.replace("\\", "\\\\").replace("\"", "\\\"")}"')
        except Exception:
            pass
        # Fallback: try unicode_escape but protect against errors
        try:
            return bytes(s, "utf-8").decode("unicode_escape")
        except Exception:
            return s  # last resort: return original

    _PROMPT_RE = re.compile(r'"prompt"\s*:\s*"(.*?)"', re.DOTALL)

    def _loose_extract_prompts(raw_text):
        if not raw_text or not isinstance(raw_text, str):
            return []
        j = _extract_json_from_text(raw_text)
        if j and isinstance(j, dict):
            out = []
            p_list = j.get("prompts")
            if isinstance(p_list, list):
                for it in p_list:
                    if isinstance(it, dict) and "prompt" in it:
                        out.append(it["prompt"])
                    elif isinstance(it, str):
                        out.append(it)
            else:
                for v in j.values():
                    if isinstance(v, dict) and "prompt" in v:
                        out.append(v["prompt"])
            if out:
                return out

        m = _PROMPT_RE.findall(raw_text)
        if m:
            return [_safe_unescape(s) for s in m]

        m2 = re.findall(r'prompt\s*[:=]\s*"(.*?)"', raw_text, re.DOTALL | re.IGNORECASE)
        if m2:
            return [_safe_unescape(s) for s in m2]

        return []


    def _make_fallbacks(definition, input_text):
        d = (definition or "").strip()
        i = (input_text or "").strip()
        p1 = f"{d} Input: {i} Provide a concise direct answer only, formatted as requested."
        p2 = f"Step-by-step: {d} Use numbered steps to extract facts from the Input then provide the final answer. Input: {i}"
        p3 = f"Context-aware: {d} Consider likely assumptions and world knowledge to disambiguate, then give the requested answer. Input: {i}"
        return [p1, p2, p3]

    # main
    definition = record.get("definition","") or ""
    input_text = record.get("input","") or record.get("inputs","") or record.get("text","") or ""
    orig_prompt = f"{definition.strip()}\n\nInput:\n{input_text.strip()}"

    gen = record.get("generation", {}) or {}
    parsed = gen.get("parsed_json")
    raw_text = gen.get("raw_text") or gen.get("raw_output") or None

    extracted = []
    sources = []

    if parsed and isinstance(parsed, dict) and not parsed.get("error_parse", False):
        p_list = parsed.get("prompts")
        if isinstance(p_list, list) and len(p_list) > 0:
            for item in p_list:
                if isinstance(item, dict) and "prompt" in item:
                    extracted.append(item["prompt"])
                    sources.append("from_parsed")
                elif isinstance(item, str):
                    extracted.append(item)
                    sources.append("from_parsed_str")

    if len(extracted) < 3 and raw_text:
        found = _loose_extract_prompts(raw_text)
        if found:
            for s in found:
                if len(extracted) >= 3:
                    break
                extracted.append(s)
                sources.append("from_raw")

    if len(extracted) < 3:
        fb = _make_fallbacks(definition, input_text)
        for s in fb:
            if len(extracted) >= 3:
                break
            extracted.append(s)
            sources.append("fallback")

    while len(extracted) < 3:
        extracted.append("")
        sources.append("empty")

    p1, p2, p3 = extracted[0], extracted[1], extracted[2]
    return orig_prompt, p1, p2, p3, ["orig"] + sources, input_text

# # Generator: stream file and yield per-record prompts (includes input_text)
# def iter_prompts_from_jsonl(path, limit=None):
#     with open(path, "r", encoding="utf-8") as f:
#         for i, line in enumerate(tqdm(f, total=None)):
#             if limit is not None and i >= limit:
#                 break
#             try:
#                 rec = json.loads(line)
#             except Exception:
#                 # skip malformed lines but continue
#                 continue
#             orig, p1, p2, p3, sources, input_text = extract_4_prompts_from_record(rec)
#             yield {
#                 "example_id": rec.get("example_id") or rec.get("id") or f"row_{i}",
#                 "source_idx": rec.get("source_idx", None),
#                 "input_text": input_text,
#                 "orig_prompt": orig,
#                 "p1": p1,
#                 "p2": p2,
#                 "p3": p3,
#                 "sources": sources
#             }

import json
from tqdm import tqdm

def iter_prompts_from_jsonl(path, limit=None):
    """
    Input JSONL lines contain:
    {
        "qid": "...",
        "original_prompt": "...",
        "generated_prompt": "..."
    }
    """
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(tqdm(f, total=None)):
            if limit is not None and i >= limit:
                break
            try:
                rec = json.loads(line)
            except Exception:
                continue

            qid = rec.get("qid") or f"row_{i}"

            yield {
                "example_id": qid,
                "qid": qid,
                "original_prompt": rec.get("original_prompt", ""),
                "generated_prompt": rec.get("generated_prompt", "")
            }



In [21]:
def build_execution_chat_template(definition: str, input_text: str):
    """
    Build a Qwen3-VL message list with two turns:
    - system: explains how to interpret Definition and Input, and instructs the model
              to use reasoning & world knowledge (without revealing chain-of-thought)
    - user: provides Definition + Input

    Returns:
        A Python list of message dicts, ready for:
            processor.apply_chat_template(messages, ...)
    """

    system_msg_text = (
        "You are a highly capable assistant. You will be given two pieces of information:\n\n"
        "1) DEFINITION — This describes *exactly* how you must produce the answer: the output\n"
        "   format, constraints, answer type (e.g., number/span/date/JSON/list), and any rules.\n\n"
        "2) INPUT — This is the specific content (passage, question, data, etc.) that the\n"
        "   DEFINITION must be applied to in order to generate the answer.\n\n"
        "Follow these hard rules:\n"
        "- Apply the DEFINITION precisely to the INPUT and produce ONLY what the DEFINITION requires.\n"
        "- Do NOT include the DEFINITION or the INPUT verbatim in your answer unless the\n"
        "  DEFINITION explicitly asks for it.\n"
        "- Do NOT produce chain-of-thought, internal reasoning steps, or other hidden reasoning.\n\n"
        "Use your reasoning abilities, domain knowledge, and world knowledge to produce the best\n"
        "possible answer that satisfies the DEFINITION. Reason internally (privately) as needed,\n"
        "but do NOT reveal your chain-of-thought. If the DEFINITION explicitly permits an explanation\n"
        "or rationale, provide a concise explanation only in the format the DEFINITION requires.\n\n"
        "If the DEFINITION is ambiguous or underspecified, make a best-effort answer using reasonable\n"
        "assumptions. **Only** include a one-line 'Assumption:' statement if the DEFINITION explicitly\n"
        "allows extra output; otherwise, do not add assumptions or commentary.\n\n"
        "Always prefer accuracy, brevity, and strict adherence to the DEFINITION's output format.\n"
    )

    user_msg_text = (
        f"Definition:\n{definition.strip()}\n\n"
        f"Input:\n{input_text.strip()}"
    )

    messages = [
        {
            "role": "system",
            "content": [
                {"type": "text", "text": system_msg_text}
            ]
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": user_msg_text}
            ]
        }
    ]

    return messages

def build_execution_chat_template(definition: str):
    """

    Returns:
        A Python list of message dicts, ready for:
            processor.apply_chat_template(messages, ...)
    """

    system_msg_text = (
        "You are a highly capable assistant. You will be given two pieces of information:\n\n"
        "1) DEFINITION — This describes *exactly* how you must produce the answer: the output\n"
        "   format, constraints, answer type (e.g., number/span/date/JSON/list), and any rules.\n\n"
        "2) INPUT — This is the specific content (passage, question, data, etc.) that the\n"
        "   DEFINITION must be applied to in order to generate the answer.\n\n"
        "Follow these hard rules:\n"
        "- Apply the DEFINITION precisely to the INPUT and produce ONLY what the DEFINITION requires.\n"
        "- Do NOT include the DEFINITION or the INPUT verbatim in your answer unless the\n"
        "  DEFINITION explicitly asks for it.\n"
        "- Do NOT produce chain-of-thought, internal reasoning steps, or other hidden reasoning.\n\n"
        "Use your reasoning abilities, domain knowledge, and world knowledge to produce the best\n"
        "possible answer that satisfies the DEFINITION. Reason internally (privately) as needed,\n"
        "but do NOT reveal your chain-of-thought. If the DEFINITION explicitly permits an explanation\n"
        "or rationale, provide a concise explanation only in the format the DEFINITION requires.\n\n"
        "If the DEFINITION is ambiguous or underspecified, make a best-effort answer using reasonable\n"
        "assumptions. **Only** include a one-line 'Assumption:' statement if the DEFINITION explicitly\n"
        "allows extra output; otherwise, do not add assumptions or commentary.\n\n"
        "Always prefer accuracy, brevity, and strict adherence to the DEFINITION's output format.\n"
    )

    # user_msg_text = (
    #     f"Definition:\n{definition.strip()}\n\n"
    #     f"Input:\n{input_text.strip()}"
    # )

    messages = [
        {
            "role": "system",
            "content": [
                {"type": "text", "text": system_msg_text}
            ]
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": definition}
            ]
        }
    ]

    return messages


In [11]:
LIMIT = 10  # change or set to None to iterate whole file

for i, item in enumerate(iter_prompts_from_jsonl(INPUT_JSONL, limit=LIMIT)):
    print(f"\n==========================")
    print(f"=== Record {i} | example_id={item['example_id']} | sources={item['sources']} ===")
    print("==========================\n")

    definition_orig = item["orig_prompt"]
    definition_p1   = item["p1"]
    definition_p2   = item["p2"]
    definition_p3   = item["p3"]

    input_text = item["input_text"]

    # Build 4 chat templates
    chat_orig = build_execution_chat_template(definition_orig, input_text)
    chat_p1   = build_execution_chat_template(definition_p1,   input_text)
    chat_p2   = build_execution_chat_template(definition_p2,   input_text)
    chat_p3   = build_execution_chat_template(definition_p3,   input_text)

    # Print the 4 chat templates
    print("---- ORIG CHAT TEMPLATE ----")
    print(chat_orig)

    print("\n---- P1 CHAT TEMPLATE ----")
    print(chat_p1)

    print("\n---- P2 CHAT TEMPLATE ----")
    print(chat_p2)

    print("\n---- P3 CHAT TEMPLATE ----")
    print(chat_p3)

    if i >= LIMIT - 1:
        break


0it [00:00, ?it/s]

KeyError: 'sources'

In [12]:
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

# 1) Load model (auto device placement)
# dtype="auto" lets HF pick bf16 on supported GPUs; you can set torch.bfloat16 explicitly if desired.
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype="auto",         # choose "auto" or torch.bfloat16
    device_map="auto",    # automatic device placement
    trust_remote_code=True,
    use_safetensors=True, # recommended by model card
)

# 2) Load processor (tokenizer + vision preprocess)
processor = AutoProcessor.from_pretrained(MODEL_ID)

# 3) Build messages (text-only). Qwen's processor expects the 'content' format shown below.
messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Explain the difference between supervised and unsupervised learning in two sentences."}
        ],
    }
]

# 4) Prepare inputs: tokenize + add generation prompt
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
)

# 5) Move inputs to model device
inputs = inputs.to(model.device)

# 6) Generate
# tune max_new_tokens / temperature / top_p etc. as you like
generated_ids = model.generate(**inputs, max_new_tokens=256)

# 7) Trim prompt part and decode (the processor has helper to decode)
# generated_ids_trimmed is list of output ids AFTER the input ids for each batch element
generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
output_texts = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)

# 8) Print
for txt in output_texts:
    print(txt)


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/2.72G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

Supervised learning uses labeled training data, where the model learns to map inputs to known outputs, enabling it to make predictions on new data. Unsupervised learning, in contrast, works with unlabeled data, aiming to uncover hidden patterns or structures without predefined target outcomes.


In [13]:

import time, json, os, traceback
from tqdm.auto import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [14]:
# -----------------------
# Config
# -----------------------
INPUT_JSONL = "/content/drive/MyDrive/AdvNLP/natural_instructions_1M_promptgeneratedNew2.jsonl"
OUTPUT_JSONL = "/content/drive/MyDrive/AdvNLP/natural_instructions_1M_inference_qwen_generate3.jsonl"



RECORDS_PER_BATCH = 3        # 1 record = 4 prompts (orig, p1, p2, p3)
MAX_NEW_TOKENS = 1000
TEMPERATURE = 0.3            # deterministic for evaluation
TOP_P = 0.9
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TRUNCATION_LEN = 4096
PRINT_DEBUG = True

GEN_KWARGS = dict(
    do_sample=True,         # deterministic
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_new_tokens=MAX_NEW_TOKENS,
    use_cache=True,
)


In [16]:


# # -----------------------
# # Helper: batched generation using the AutoProcessor
# # -----------------------
def generate_batch_messages(messages_batch, processor, model, gen_kwargs=None, debug_prefix="batch"):
    """
    messages_batch: list of conversations, where each conversation is a list of message dicts:
      [
        [ {"role":"user","content":[...]}, {"role":"assistant", ...}, ... ],   # conv 1
        [ {"role":"user","content":[...]} ],                                  # conv 2
        ...
      ]
    Returns: list of decoded strings (one per conversation) - only the generated portion
    """
    gen_kwargs = {} if gen_kwargs is None else dict(gen_kwargs)

    # apply chat template + tokenize + pad for the whole batch
    inputs = processor.apply_chat_template(
        messages_batch,
        tokenize=True,
        add_generation_prompt=True,
        padding=True,               # IMPORTANT: pad the batch
        truncation=True,
        max_length=TRUNCATION_LEN,
        return_tensors="pt",
        return_dict=True,
    )

    # Move tensors to model device
    device = next(model.parameters()).device
    inputs = inputs.to(device)

    if PRINT_DEBUG:
        print(f"[{debug_prefix}] input_ids shape: {inputs.input_ids.shape} attention_mask: {getattr(inputs, 'attention_mask', None).shape}")

    # generation defaults (safe)
    defaults = dict(
        input_ids=inputs.input_ids,
        attention_mask=getattr(inputs, "attention_mask", None),
        max_new_tokens=GEN_KWARGS.get("max_new_tokens", 256),
        eos_token_id=processor.tokenizer.eos_token_id,
        pad_token_id=processor.tokenizer.eos_token_id,
        do_sample=GEN_KWARGS.get("do_sample", False),
        temperature=GEN_KWARGS.get("temperature", 1.0),
        top_k=GEN_KWARGS.get("top_k", None),
        top_p=GEN_KWARGS.get("top_p", None),
        use_cache=True,
    )
    # allow overrides
    defaults.update(gen_kwargs)

    # generate
    t0 = time.time()
    with torch.no_grad():
        if torch.cuda.is_available(): torch.cuda.synchronize()
        out_ids = model.generate(**defaults)
        if torch.cuda.is_available(): torch.cuda.synchronize()
    t1 = time.time()

    if PRINT_DEBUG:
        print(f"[{debug_prefix}] gen_time={t1-t0:.2f}s")

    # Trim prompt portion and decode per batch element
    trimmed_list = []
    for in_ids, out in zip(inputs.input_ids, out_ids):
        prompt_len = in_ids.shape[0]
        gen_ids = out[prompt_len:]
        # convert to cpu list for batch_decode
        trimmed_list.append(gen_ids.cpu())

    # processor.batch_decode expects list of torch tensors or lists
    decoded = processor.batch_decode(
        trimmed_list,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    # decoded is a list of strings, one per conversation
    return decoded


In [26]:
import os
import json
import time
from tqdm import tqdm

PRINT_DEBUG = True
INPUT_JSONL = "/content/drive/MyDrive/AdvNLP/generated_prompts_pre_finetune_with_iptext.jsonl"
OUTPUT_JSONL = "/content/drive/MyDrive/AdvNLP/generated_prompts_pre_finetune_output.jsonl"
RECORDS_PER_BATCH = 8
TARGET_COUNT = None  # or some int


def read_processed_ids(output_jsonl_path):
    processed = set()
    if not os.path.exists(output_jsonl_path):
        return processed
    with open(output_jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
                if "qid" in obj:
                    processed.add(obj["qid"])
            except:
                continue
    return processed


def genprompts(
    input_jsonl=INPUT_JSONL,
    output_jsonl=OUTPUT_JSONL,
    records_per_batch=RECORDS_PER_BATCH,
    target_count=TARGET_COUNT,
):
    if not os.path.exists(input_jsonl):
        raise FileNotFoundError(f"Input not found: {input_jsonl}")

    print(f"Starting/resuming.\nInput = {input_jsonl}\nOutput = {output_jsonl}")

    processed_ids = read_processed_ids(output_jsonl)
    print(f"Found {len(processed_ids)} already-processed qids")

    os.makedirs(os.path.dirname(output_jsonl), exist_ok=True)
    out_f = open(output_jsonl, "a", encoding="utf-8")

    total_written = 0
    newly_written = 0
    batch_records = []
    start_time = time.time()

    # Count lines for tqdm (safe fallback)
    try:
        with open(input_jsonl, "r", encoding="utf-8") as fcnt:
            total_lines = sum(1 for _ in fcnt)
    except:
        total_lines = None

    try:
        for rec in tqdm(iter_prompts_from_jsonl(input_jsonl), total=total_lines):

            qid = rec["qid"]
            if qid in processed_ids:
                continue

            batch_records.append(rec)

            if len(batch_records) >= records_per_batch:
                # --------- BUILD CHAT BATCH ---------
                messages_batch = []
                mapping = []

                for ridx, r in enumerate(batch_records):
                    chat_orig = build_execution_chat_template(r["original_prompt"])
                    chat_gen  = build_execution_chat_template(r["generated_prompt"])

                    if not isinstance(chat_orig, list) or not isinstance(chat_gen, list):
                        raise ValueError("build_execution_chat_template must return list of messages")

                    messages_batch.extend([chat_orig, chat_gen])
                    mapping.extend([(ridx, "original"), (ridx, "generated")])

                # --------- RUN GENERATION ---------
                try:
                    decoded = generate_batch_messages(
                        messages_batch, processor, model,
                        gen_kwargs=GEN_KWARGS,
                        debug_prefix=f"batch_{newly_written}"
                    )
                except Exception as e:
                    print("Generation batch error:", repr(e))
                    decoded = [""] * len(messages_batch)

                # --------- MAP BACK ---------
                per_rec_out = [{"original_response": "", "generated_response": ""} for _ in batch_records]

                for txt, (ridx, slot) in zip(decoded, mapping):
                    key = f"{slot}_response"
                    per_rec_out[ridx][key] = txt.strip()

                # --------- WRITE OUTPUT ---------
                for local_idx, r in enumerate(batch_records):

                    rec_out = {
                        "qid": r["qid"],
                        "original_prompt": r["original_prompt"],
                        "original_prompt_response": per_rec_out[local_idx]["original_response"],

                        "generated_prompt": r["generated_prompt"],
                        "generated_prompt_response": per_rec_out[local_idx]["generated_response"],

                        "processed_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
                    }

                    out_f.write(json.dumps(rec_out, ensure_ascii=False) + "\n")
                    out_f.flush()

                    processed_ids.add(r["qid"])
                    newly_written += 1
                    total_written += 1

                    if target_count is not None and newly_written >= target_count:
                        break

                batch_records = []

                if target_count is not None and newly_written >= target_count:
                    break

        # --------- PROCESS FINAL PARTIAL BATCH ---------
        if batch_records and (target_count is None or newly_written < target_count):

            messages_batch = []
            mapping = []

            for ridx, r in enumerate(batch_records):
                chat_orig = build_execution_chat_template(r["original_prompt"])
                chat_gen  = build_execution_chat_template(r["generated_prompt"])

                messages_batch.extend([chat_orig, chat_gen])
                mapping.extend([(ridx, "original"), (ridx, "generated")])

            try:
                decoded = generate_batch_messages(
                    messages_batch, processor, model,
                    gen_kwargs=GEN_KWARGS,
                    debug_prefix=f"final_{newly_written}"
                )
            except Exception as e:
                print("Final batch error:", repr(e))
                decoded = [""] * len(messages_batch)

            per_rec_out = [{"original_response": "", "generated_response": ""} for _ in batch_records]

            for txt, (ridx, slot) in zip(decoded, mapping):
                per_rec_out[ridx][f"{slot}_response"] = txt.strip()

            for local_idx, r in enumerate(batch_records):
                rec_out = {
                    "qid": r["qid"],
                    "original_prompt": r["original_prompt"],
                    "original_prompt_response": per_rec_out[local_idx]["original_response"],
                    "generated_prompt": r["generated_prompt"],
                    "generated_prompt_response": per_rec_out[local_idx]["generated_response"],
                    "processed_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
                }

                out_f.write(json.dumps(rec_out, ensure_ascii=False) + "\n")
                out_f.flush()

                processed_ids.add(r["qid"])
                newly_written += 1
                total_written += 1

                if target_count is not None and newly_written >= target_count:
                    break

    finally:
        out_f.close()

    elapsed = time.time() - start_time
    print(f"Done. total_written={total_written}, newly_written={newly_written}, elapsed={elapsed:.1f}s")
    return total_written


In [27]:
# # Example usage:
genprompts(input_jsonl=INPUT_JSONL, output_jsonl=OUTPUT_JSONL, records_per_batch=RECORDS_PER_BATCH, target_count=TARGET_COUNT)


Starting/resuming.
Input = /content/drive/MyDrive/AdvNLP/generated_prompts_pre_finetune_with_iptext.jsonl
Output = /content/drive/MyDrive/AdvNLP/generated_prompts_pre_finetune_output.jsonl
Found 0 already-processed qids


  0%|          | 0/552 [00:00<?, ?it/s]
0it [00:00, ?it/s]

[batch_0] input_ids shape: torch.Size([16, 1064]) attention_mask: torch.Size([16, 1064])


  1%|▏         | 8/552 [00:18<20:40,  2.28s/it]
8it [00:18,  2.28s/it]

[batch_0] gen_time=18.21s
[batch_8] input_ids shape: torch.Size([16, 1028]) attention_mask: torch.Size([16, 1028])


...

 99%|█████████▊| 544/552 [15:32<00:15,  1.88s/it]
544it [15:32,  1.88s/it]

[batch_536] gen_time=17.32s
[batch_544] input_ids shape: torch.Size([16, 687]) attention_mask: torch.Size([16, 687])


100%|██████████| 552/552 [15:49<00:00,  1.97s/it]
552it [15:49,  1.72s/it]
100%|██████████| 552/552 [15:49<00:00,  1.72s/it]

[batch_544] gen_time=17.41s
Done. total_written=552, newly_written=552, elapsed=950.3s


552